In [1]:
import pandas as pd
from modules.data_hourly_preprocessing import DataCleaner

In [2]:
TARGET_COL ="pm25"
HORIZON = 24

In [3]:
fecheck_df= pd.read_parquet(r"D:\pypipeline\data\processed\hourly\us_paro_hourly\test_processed.parquet")

features_exclude = [f"{TARGET_COL}_plus_{i}h" for i in range(1, HORIZON + 1)] + \
                       [f"o3_plus_{i}h" for i in range(1, HORIZON + 1)] + \
                       [f"pm25_plus_{i}h" for i in range(1, HORIZON + 1)] + \
                       ["segment_id", "imputation_confidence"]

In [4]:
fecheck_df =fecheck_df.drop(columns=features_exclude)

In [5]:
fecheck_df.head()

,pm25,o3,pm25_target,o3_target,hour,day_of_week,is_weekend,is_night,pm25_missing,o3_missing,...,o3_roll_6,pm25_roll_12,o3_roll_12,pm25_roll_24,o3_roll_24,pm25_slope_3h,o3_slope_3h,pm25_slope_12h,o3_slope_12h,pm25_o3_ratio
date,,,,,,,,,,,,,,,,,,,,,
2020-08-04 10:00:00+05:45,16.0,0.039,16.0,0.039,10,1,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,410.25641
2020-08-04 11:00:00+05:45,16.0,0.039,16.0,0.039,11,1,0,0,0,0,...,0.039,16.0,0.039,16.0,0.039,NaN,NaN,NaN,NaN,410.25641
2020-08-04 12:00:00+05:45,NaN,NaN,NaN,NaN,12,1,0,0,1,1,...,0.039,16.0,0.039,16.0,0.039,NaN,NaN,NaN,NaN,NaN
2020-08-04 13:00:00+05:45,NaN,NaN,NaN,NaN,13,1,0,0,1,1,...,0.039,16.0,0.039,16.0,0.039,NaN,NaN,NaN,NaN,NaN
2020-08-04 14:00:00+05:45,NaN,NaN,NaN,NaN,14,1,0,0,1,1,...,0.039,16.0,0.039,16.0,0.039,NaN,NaN,NaN,NaN,NaN


In [6]:
df = pd.read_csv(r"D:\pypipeline\data\raw\pred_ingestion\us_paro\us_paro_hourly.csv")

In [7]:
df.drop(columns=["pm10"], inplace=True)

In [8]:
df["date"] = pd.to_datetime(df["date"])
df.set_index("date", inplace=True)

In [9]:
clean = DataCleaner()
df_1 = clean.add_time_features(df)
df_2 = clean.add_missing_flags(df_1)
df_2["was_imputed"]=0
df_3 = clean.add_gap_length(df_2)
df_3 = clean.add_segmentation(df_3)

df_4 = clean.engineer_features(df_3)

2026-02-13 21:18:23 | INFO | modules.data_hourly_preprocessing | Adding time-based features
2026-02-13 21:18:23 | INFO | modules.data_hourly_preprocessing | Adding missing-value flags
2026-02-13 21:18:23 | INFO | modules.data_hourly_preprocessing | Computing gap length features
2026-02-13 21:18:23 | INFO | modules.data_hourly_preprocessing | Gap length features added
2026-02-13 21:18:23 | INFO | modules.data_hourly_preprocessing | Adding segment identifiers
2026-02-13 21:18:23 | INFO | modules.data_hourly_preprocessing | Segmentation complete
2026-02-13 21:18:23 | INFO | modules.data_hourly_preprocessing | Engineering features
2026-02-13 21:18:23 | INFO | modules.data_hourly_preprocessing | Adding lag features
2026-02-13 21:18:23 | INFO | modules.data_hourly_preprocessing | Adding rolling window features
2026-02-13 21:18:23 | INFO | modules.data_hourly_preprocessing | Adding slope and ratio features
2026-02-13 21:18:23 | INFO | modules.data_hourly_preprocessing | Feature engineering co

In [ ]:
df_4.head()


,o3,pm25,hour,day_of_week,is_weekend,is_night,pm25_missing,o3_missing,was_imputed,pm25_gap_length,...,o3_roll_6,pm25_roll_12,o3_roll_12,pm25_roll_24,o3_roll_24,pm25_slope_3h,o3_slope_3h,pm25_slope_12h,o3_slope_12h,pm25_o3_ratio
date,,,,,,,,,,,,,,,,,,,,,
2026-02-09 17:45:00,111.87,124.68,17,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.114508
2026-02-09 18:45:00,99.74,130.26,18,0,0,0,0,0,0,0,...,111.8700,124.6800,111.8700,124.6800,111.8700,NaN,NaN,NaN,NaN,1.305996
2026-02-09 19:45:00,90.10,137.91,19,0,0,0,0,0,0,0,...,105.8050,127.4700,105.8050,127.4700,105.8050,6.615,-10.885,6.615,-10.885,1.530633
2026-02-09 20:45:00,83.94,145.22,20,0,0,0,0,0,0,0,...,100.5700,130.9500,100.5700,130.9500,100.5700,7.480,-7.900,6.927,-9.343,1.730045
2026-02-09 21:45:00,76.66,149.80,21,0,0,0,0,0,0,0,...,96.4125,134.5175,96.4125,134.5175,96.4125,5.945,-6.720,6.520,-8.622,1.954083


In [21]:
import mlflow
mlflow.set_tracking_uri("http://localhost:5000")
model = mlflow.pyfunc.load_model("models:/hourly_pm25_24h_service/latest")

In [26]:
df_4

,o3,pm25,hour,day_of_week,is_weekend,is_night,pm25_missing,o3_missing,was_imputed,pm25_gap_length,...,o3_roll_6,pm25_roll_12,o3_roll_12,pm25_roll_24,o3_roll_24,pm25_slope_3h,o3_slope_3h,pm25_slope_12h,o3_slope_12h,pm25_o3_ratio
0,111.87,124.68,17,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.114508
1,99.74,130.26,18,0,0,0,0,0,0,0,...,111.870000,124.680000,111.870000,124.680000,111.870000,NaN,NaN,NaN,NaN,1.305996
2,90.10,137.91,19,0,0,0,0,0,0,0,...,105.805000,127.470000,105.805000,127.470000,105.805000,6.615,-10.885,6.615000,-10.885000,1.530633
3,83.94,145.22,20,0,0,0,0,0,0,0,...,100.570000,130.950000,100.570000,130.950000,100.570000,7.480,-7.900,6.927000,-9.343000,1.730045
4,76.66,149.80,21,0,0,0,0,0,0,0,...,96.412500,134.517500,96.412500,134.517500,96.412500,5.945,-6.720,6.520000,-8.622000,1.954083
5,64.53,157.65,22,0,0,0,0,0,0,0,...,92.462000,137.574000,92.462000,137.574000,92.462000,6.215,-9.705,6.593714,-8.917143,2.443050
6,53.74,164.13,23,0,0,0,0,0,0,0,...,87.806667,140.920000,87.806667,140.920000,87.806667,7.165,-11.460,6.607857,-9.223214,3.054150
7,44.52,167.12,0,1,0,1,0,0,0,0,...,78.118333,144.235714,82.940000,144.235714,82.940000,4.735,-10.005,6.312262,-9.350476,3.753819
8,36.58,171.74,1,1,0,1,0,0,0,0,...,68.915000,147.096250,78.137500,147.096250,78.137500,3.805,-8.580,6.061500,-9.315833,4.694915
9,29.77,176.30,2,1,0,1,0,0,0,0,...,59.995000,149.834444,73.520000,149.834444,73.520000,4.590,-7.375,5.851939,-9.161515,5.922069


In [28]:
fin = df_4.iloc[1,:]

In [30]:
fin

o3                 99.740000
pm25              130.260000
hour               18.000000
day_of_week         0.000000
is_weekend          0.000000
                     ...    
pm25_slope_3h            NaN
o3_slope_3h              NaN
pm25_slope_12h           NaN
o3_slope_12h             NaN
pm25_o3_ratio       1.305996
Name: 1, Length: 73, dtype: float64

In [29]:
model.predict(fin)

MlflowException: Failed to enforce schema of data '<mlflow.pyfunc.model.PythonModelContext object at 0x000002091E2BF170>' with schema '['pm25': double (optional), 'o3': double (optional), 'hour': integer (required), 'day_of_week': integer (required), 'is_weekend': long (required), 'is_night': long (required), 'pm25_missing': long (required), 'o3_missing': long (required), 'pm25_gap_length': long (required), 'o3_gap_length': long (required), 'was_imputed': double (optional), 'hour_sin': double (required), 'hour_cos': double (required), 'pm25_lag_1': double (optional), 'o3_lag_1': double (optional), 'pm25_lag_2': double (optional), 'o3_lag_2': double (optional), 'pm25_lag_3': double (optional), 'o3_lag_3': double (optional), 'pm25_lag_4': double (optional), 'o3_lag_4': double (optional), 'pm25_lag_5': double (optional), 'o3_lag_5': double (optional), 'pm25_lag_6': double (optional), 'o3_lag_6': double (optional), 'pm25_lag_7': double (optional), 'o3_lag_7': double (optional), 'pm25_lag_8